# Aralin 11 - Protokol ng Ahente-sa-Ahente (A2A)


## Pagsasaayos


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ano ang A2A na Protokol?

Ang **Agent-to-Agent (A2A) na protokol** ay isang bukas na pamantayan na nagpapahintulot sa mga AI agent na magkomunika,
maghanap ng isa't isa, at makipagtulungan — kahit na sila ay binuo sa iba't ibang mga framework o naka-host
ng iba't ibang serbisyo.

Mga pangunahing konsepto:

- **Discovery** – Naglalathala ang mga agent ng isang *Agent Card* na naglalarawan ng kanilang mga kakayahan, na nagpapadali
  para sa ibang mga agent (o mga orchestrator) na mahanap ang tamang espesyalista para sa isang gawain.
- **Message Passing** – Nagpapalitan ang mga agent ng mga istrukturadong mensahe sa pamamagitan ng isang karaniwang protokol, kaya
  ang isang kahilingan mula sa isang agent ay mauunawaan at maisasakatuparan ng isa pa kahit ano man ang panloob
  implementasyon.
- **Task Lifecycle** – Inilalarawan ng A2A ang mga estado tulad ng *submitted*, *working*, *completed*, at
  *failed*, na nagbibigay sa orchestrator ng buong pananaw kung paano umuunlad ang isang ipinaupang task.

Sa leksyon na ito ginagaya natin ang kolaborasyon na istilo ng A2A sa pamamagitan ng pagkakabit ng tatlong espesyalistang ahente sa paglalakbay
sa isang workflow kung saan ang bawat ahente ay nag-aambag ng kanyang kadalubhasaan at ipinapasa ang mga resulta sa susunod.


## Paglikha ng Mga Espesyalistang Ahente sa Paglalakbay


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Pagtutulungan ng Maramihang Ahente sa pamamagitan ng Daloy ng Trabaho

Kinokonekta namin ang tatlong ahente sa isang sunud-sunod na daloy ng trabaho na sumasalamin sa A2A na pagpapasa ng mensahe:

1. **CurrencyExchangeAgent** tumatanggap ng kahilingan ng gumagamit at nagbibigay ng gabay tungkol sa palitan ng pera.
2. **ActivityPlannerAgent** tumatanggap ng pinagyamang konteksto at nagdaragdag ng mga rekomendasyon para sa mga aktibidad.
3. **TravelManagerAgent** pinagsasama ang parehong input upang makabuo ng pangwakas na buod ng paglalakbay.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pag-unawa sa A2A sa Produksyon

Sa isang kapaligiran ng produksyon, nagbubukas ang protocol na A2A ng makapangyarihang mga cross-service na senaryo:

| Capability | Description |
|---|---|
| **Interoperabilidad sa pagitan ng mga framework** | Ang isang agent na binuo gamit ang isang framework ay maaaring ipasa ang mga gawain sa isang agent na binuo gamit ang alinmang iba pang sumusunod sa A2A na framework, na nagpapahintulot ng tunay na interoperability sa pagitan ng mga organisasyon. |
| **Mga hangganan ng serbisyo** | Ang mga agent ay maaaring umiral sa magkakahiwalay na microservices, mga rehiyon ng cloud, o kahit sa iba't ibang mga organisasyon habang patuloy na nakikipagtulungan nang walang putol. |
| **Dinamikong pagtuklas** | Maaaring mag-query ang isang orchestrator sa isang rehistro ng Agent Card habang tumatakbo upang hanapin ang pinaka-angkop na espesyalista para sa isang partikular na sub-gawain. |
| **Streaming & push notifications** | Sinusuportahan ng A2A ang Server-Sent Events (SSE) para sa real-time na mga update sa progreso at mga push notification para sa mga matagal na gawain. |

Ang workflow na binuo namin sa itaas ay isang pinasimpleng, in-process na bersyon ng pattern na ito. Sa isang tunay na deployment, bawat agent ay magbibigay ng isang HTTP endpoint, maglalathala ng isang Agent Card, at makikipag-ugnayan sa pamamagitan ng A2A JSON-RPC protocol.


## Buod

Sa leksyong ito natutunan mo:

1. **Ano ang protokol ng A2A** — isang bukas na pamantayan para sa pagtuklas, pagpapadala ng mensahe, at pamamahala ng gawain sa pagitan ng mga ahente.
2. **Paano lumikha ng mga espesyalisadong ahente** — a Currency Exchange agent, an Activity Planner agent, and a Travel Manager orchestrator.
3. **Paano iugnay ang mga ahente sa isang workflow** — gamit ang `WorkflowBuilder` upang i-modelo ang sunud-sunod na pagpapadala ng mensahe sa pagitan ng mga ahente.
4. **Paano gumagana ang A2A sa produksyon** — pinapagana ang pakikipagtulungan sa pagitan ng iba't ibang framework at serbisyo gamit ang dinamikong pagtuklas at streaming na mga pag-update.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Paunawa:
Ang dokumentong ito ay isinalin gamit ang serbisyong pagsasalin na pinapagana ng AI na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagaman sinisikap naming maging tumpak, pakitandaan na ang mga awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o di-tumpak na impormasyon. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na awtoritatibong pinagmulan. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasaling gawa ng tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring magmula sa paggamit ng salin na ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
